# Metaclasses – Practice Exercises

This notebook contains hands-on exercises to practice Python **metaclasses**.

The progression goes from basics (`type` and dynamic class creation) to practical patterns like registries, validation contracts, and a small declarative mini-DSL.

For each exercise:

- Read the description carefully.
- Implement the TODO parts in the provided code cell.
- Run the demo tests at the bottom of the cell.


### Contents

- [Exercise 1 – Dynamic Classes with `type`](#exercise-1--dynamic-classes-with-type)
- [Exercise 2 – Logging Metaclass](#exercise-2--logging-metaclass)
- [Exercise 3 – Auto-Register Subclasses](#exercise-3--auto-register-subclasses)
- [Exercise 4 – Enforce Required Class Attributes](#exercise-4--enforce-required-class-attributes)
- [Exercise 5 – Declarative Mini-DSL](#exercise-5--declarative-mini-dsl)
- [Exercise 6 – Metaclass + ABC Contract](#exercise-6--metaclass--abc-contract)
- [Exercise 7 – Validate Instance Creation in `__call__`](#exercise-7--validate-instance-creation-in-__call__)
- [Exercise 8 – Singleton Metaclass](#exercise-8--singleton-metaclass)
- [Exercise 9 – Strategy Registry Use-Case](#exercise-9--strategy-registry-use-case)
- [Exercise 10 – When Not to Use Metaclasses](#exercise-10--when-not-to-use-metaclasses)


## Exercise 1 – Dynamic Classes with `type`

**Goal**: Use `type(name, bases, attrs)` to build a class dynamically.

**Requirements**:

- Create a dynamic class `DynamicOrder` inheriting from `object`.
- Give it a class attribute `kind = "dynamic"`.
- Instantiate it and verify `.kind` exists.

**Hint**: The second argument is a tuple of base classes.


In [15]:
# Exercise 1 – create DynamicOrder via type

DynamicOrder = type("DynamicOrder", (object,), {"kind": "dynamic"})
print(f"Class repr: \"{repr(DynamicOrder)}\"")
print(f"Class name: \"{DynamicOrder.__name__}\"")
print(f"Class type: \"{type(DynamicOrder)}\"")
ParentClass = DynamicOrder.__base__
print(f"Parent repr: \"{repr(ParentClass)}\"")
print(f"Parent name: \"{ParentClass.__name__}\"")
print(f"Parent name: \"{type(ParentClass)}\"")
print(f"Class kind: \"{DynamicOrder.kind}\"")

Class repr: "<class '__main__.DynamicOrder'>"
Class name: "DynamicOrder"
Class type: "<class 'type'>"
Parent repr: "<class 'object'>"
Parent name: "object"
Parent name: "<class 'type'>"
Class kind: "dynamic"


## Exercise 2 – Logging Metaclass

**Goal**: Intercept class creation with `__new__`.

**Requirements**:

- Create `LoggingMeta`.
- In `__new__`, print the class name being created.
- Create a base class `BaseModel(metaclass=LoggingMeta)`.
- Subclass it and verify the print happens.

**Hint**: `__new__(mcls, name, bases, namespace)` is called during class creation.


In [56]:
# Exercise 2 – implement LoggingMeta

from typing import Any

class MetaClass(type):
    def __new__(meta, name, bases, namespace, **kwargs):
        print(f"Creating class \"{name}\" from metaclass \"{meta.__name__}\"")
        attrs_str = str.join("\n", [f" - {repr(attr)}" for attr in namespace])
        bases_str = str.join("\n", [f" - {repr(base)}" for base in bases])
        if (len(bases) == 0): bases_str = "       N/A"
        print(f"...Bases:\n{bases_str}\n...Attributes:\n{attrs_str}")
        return super().__new__(meta, name, bases, namespace, **kwargs)

class BaseClass(metaclass = MetaClass):
    """Base for classes using MetaClass."""
    arg1: Any = None
    arg2: Any = None
    def __repr__(self):
        return f"{self.arg1} {self.arg2}"

base_model = BaseClass()
print(f"Instance of \"BaseClass\": \"{repr(base_model)}\"")

Creating class "BaseClass" from metaclass "MetaClass"
...Bases:
       N/A
...Attributes:
 - '__module__'
 - '__qualname__'
 - '__firstlineno__'
 - '__annotations__'
 - '__doc__'
 - 'arg1'
 - 'arg2'
 - '__repr__'
 - '__static_attributes__'
Instance of "BaseClass": "None None"


## Exercise 3 – Auto-Register Subclasses

**Goal**: Build a registry of subclasses automatically.

**Requirements**:

- Create `RegistryMeta` metaclass.
- Maintain `registry: dict[str, type]` as a class attribute on the base class.
- Every subclass should define a unique `key` string.
- Register subclasses during `__init__` or `__new__`.
- Provide a helper `create(key, *args, **kwargs)` on the base class.

**Hint**: Look at `namespace` for `key`.


In [ ]:
# Exercise 3 – implement RegistryMeta and create() helper

from typing import ClassVar, Dict, Type

class MetaClass(type):
    _registry: Dict[str, Type["BaseClass"]] = dict()
    def __new__(meta, name, bases, namespace, **kwargs):
        cls = super().__new__(meta, name, bases, namespace, **kwargs)
        name = getattr(cls, "name", name)
        if name not in meta._registry: meta._registry[name] = cls 
        else: raise ValueError(f"Duplicate class: \"{name}\"")
        key = 
        @classmethod
        def 
        
        return cls
    
    @classmethod
    def get_class(mcls, name: str) -> Type["BaseClass"]:
        return mcls._registry[name]

    @classmethod
    def available_classes(mcls) -> Dict[str, Type["BaseClass"]]:
        return dict(mcls._registry)

class BaseClass(metaclass = MetaClass):
    key: str
    def create(self, key, *args, **kwargs):
        cls = MetaClass.get_class(key)
        return cls(*args, **kwargs)

# Solution for Exercise 3 – Auto-Register Subclasses

class RegistryMeta(type):
    def __new__(mcs, name, bases, namespace, **kwargs):
        cls = super().__new__(mcs, name, bases, namespace, **kwargs)
        # If this is the root base class, install the registry and helper
        if not any(hasattr(base, 'registry') for base in bases):
            cls.registry = {}
            @classmethod
            def create(cls, key, *args, **kwargs):
                try:
                    subcls = cls.registry[key]
                except KeyError:
                    raise KeyError(f"No subclass registered with key: {key!r}")
                return subcls(*args, **kwargs)
            cls.create = create
        else:
            key = namespace.get('key', None)
            if key is None:
                # allow base abstract/intermediate classes to omit key
                pass
            else:
                base = next((b for b in bases if hasattr(b, 'registry')), None)
                if base:
                    if key in base.registry:
                        raise ValueError(f"Duplicate key '{key}' in registry for {base.__name__}")
                    base.registry[key] = cls
        return cls

# Example usage:

class RegistryBase(metaclass=RegistryMeta):
    pass

# Example subclass registration:
class Alpha(RegistryBase):
    key = "alpha"
    def __init__(self, val): self.val = val

class Beta(RegistryBase):
    key = "beta"
    def __init__(self, v): self.v = v

# Test helper:
# instance = RegistryBase.create("alpha", 123)
# print(instance)



## Exercise 4 – Enforce Required Class Attributes

**Goal**: Validate contracts at class creation time.

**Requirements**:

- Create `RequireKeyMeta` metaclass.
- For any subclass of `KeyedStrategy`, enforce that it defines `key`.
- If `key` is missing, raise `TypeError`.

**Hint**: Metaclass `__new__` can inspect the `namespace`.


In [ ]:
# Exercise 4 – implement RequireKeyMeta

## Exercise 5 – Declarative Mini-DSL

**Goal**: Turn class-level declarations into runtime metadata.

**Requirements**:

- Create `Field` objects that store `name` and `default`.
- Create `DSLModelMeta` metaclass.
- For a subclass of `DSLModel`, collect all `Field` instances defined as class attributes into a dict `fields`.
- Create `DSLModel` base class using `DSLModelMeta`.
- Demonstrate with a `OrderSpec` model having 2-3 fields.

**Hint**: Iterate over `namespace.items()` in the metaclass.


In [ ]:
# Exercise 5 – implement Field + DSLModelMeta

## Exercise 6 – Metaclass + ABC Contract

**Goal**: Combine metaclass behavior with ABC abstract methods.

**Requirements**:

- Create `BaseABC(ABC)` with an abstract `run()` method.
- Create a metaclass `CheckRunMeta` that checks a class defines `run` as an attribute.
- Create `class Constrained(BaseABC, metaclass=CheckRunMeta)`.
- Demonstrate:
- A subclass implementing `run()` works.
- A subclass missing `run` raises an error during class creation.

**Hint**: If you combine metaclasses, the metaclass must be compatible with ABC's metaclass behavior.


In [ ]:
# Exercise 6 – implement CheckRunMeta + ABC interaction

from abc import ABC, abstractmethod, ABCMeta

## Exercise 7 – Validate Instance Creation in `__call__`

**Goal**: Override metaclass `__call__` to validate constructor arguments.

**Requirements**:

- Create `NonNegativeMeta`.
- In `__call__`, if the class being instantiated receives `value=<...>`, require it to be non-negative.
- Instantiate a `Wrapper` class with `value` and show it works for valid input and fails otherwise.

**Hint**: `__call__` runs every time you create an instance.


In [ ]:
# Exercise 7 – implement NonNegativeMeta

## Exercise 8 – Singleton Metaclass

**Goal**: Ensure only one instance exists per class.

**Requirements**:

- Implement `SingletonMeta`.
- Using it, create `class Config(metaclass=SingletonMeta)`.
- Instantiate `Config()` twice and show they are the same object (`is`).

**Hint**: Cache instances in a dict on the metaclass.


In [ ]:
# Exercise 8 – implement SingletonMeta

## Exercise 9 – Strategy Registry Use-Case

**Goal**: Put Exercises 3/4 together into a practical registry pattern.

**Requirements**:

- Define a base `Strategy` using a registry metaclass (like `RegistryMeta`).
- Implement two strategies with different `key` values.
- Add a `run_strategy(key, *args, **kwargs)` function that creates and runs the strategy.
- Demonstrate calling `run_strategy()` for both keys.

**Hint**: Keep strategy behavior trivial; focus on registry wiring.


In [ ]:
# Exercise 9 – registry use-case

## Exercise 10 – When Not to Use Metaclasses

**Goal**: Practice identifying an unnecessary metaclass pattern.

**Requirements**:

- Implement the same behavior from Exercise 8 (singleton) **without** metaclasses, using a closure or a decorator.
- Compare the two approaches in a short comment: what do you gain/lose?
- Demo: create `Config2` using the non-metaclass approach; show that `Config2()` is singleton.

**Hint**: A decorator that caches the instance is often enough.


In [ ]:
# Exercise 10 – singleton without metaclasses